In [61]:
!uv add 'markitdown[pdf]'

Resolved 137 packages in 4ms
Checked 133 packages in 108ms


In [ ]:
from markitdown import MarkItDown
from pathlib import Path

md = MarkItDown()

books_dir = Path("books")
output_dir = Path("books_text")
output_dir.mkdir(exist_ok=True)

for each_file in books_dir.glob("*.pdf"):
    print("Converting", each_file)

    result = md.convert(str(each_file))
    text = result.text_content

    output_path = output_dir / (each_file.stem + ".md")
    output_path.write_text(text, encoding="utf-8")

Converting books/think-c.pdf


Converting books/think-python-2e.pdf
Converting books/think-dsp.pdf
Converting books/physical-modeling-in-matlab.pdf
Converting books/think-os.pdf
Converting books/think-java-2e.pdf
Converting books/think-complexity-2e.pdf


In [63]:
text = (output_dir / "think-python-2e.md").read_text(encoding="utf-8")
print(len(text.splitlines()))

9703


In [64]:
! uv add gitsource
from gitsource import chunk_documents

# output_dir = Path("book_text")
# documents = []

# for md_file in output_dir.glob("*.md"):
#     text = md_file.read_text(encoding="utf-8")
#     lines = text.splitlines()

#     non_empty_lines = []
#     for each_line in lines:
#         if each_line.strip():
#             non_empty_lines.append(each_line)

#     document = {
#         "source": md_file.name,
#         "content": non_empty_lines
#     }
#     documents.append(document)

# print(len(documents))

from pathlib import Path
from gitsource import chunk_documents

output_dir = Path("books_text")
documents = []

for md_file in output_dir.glob("*.md"):
    lines = md_file.read_text(encoding="utf-8").splitlines()
    documents.append({
        "source": md_file.name,
        "content": [line for line in lines if line.strip()],
    })

think_python = [d for d in documents if d["source"] == "think-python-2e.md"]

chunks = chunk_documents(think_python, size=100, step=50)
print(len(chunks))


Resolved 137 packages in 2ms
Checked 133 packages in 1ms


189


In [65]:
len(chunks)

189

In [66]:
from minsearch import Index

all_chunks = chunk_documents(documents, size=100, step=50)

def prepare_documents(chunks):
    prepared = []
    for chunk in chunks:
        doc = chunk.copy()
        content = doc["content"]
        if isinstance(content, list):
            doc["content"] = "\n".join(content)
        prepared.append(doc)
    return prepared

index_documents = prepare_documents(all_chunks)

index = Index(text_fields=["content"])
index.fit(index_documents)

print(len(index_documents))

988


In [67]:
results = index.search("python function definition", num_results=5)

In [68]:
results

[{'start': 1800,
  'content': "| ------ | ------------ | --- | --- |\n| total  | = total +    | a;  |     |\nend\nans = total\nIfyouwereusinganyofthosevariablenamesbeforecallingthisscript, youmightbesurprised\nto find, after running the script, that their values had changed. If you have two scripts that\n48 Functions\nuse the same variable names, you might find that they work separately and then break when\nyou try to combine them. This kind of interaction is called a collision.\nname\nAs the number of scripts you write increases, and they get longer and more complex, name\ncollisions become more of a problem. Avoiding this problem is one of several motivations for\nfunctions.\n| 5.2 | Defining | Functions |     |     |\n| --- | -------- | --------- | --- | --- |\nA function is like a script, except that each function has its own workspace, so any variables\ndefined inside a function only exist while the function is running and don’t interfere with\nvariables in other workspaces, even 

In [69]:
import json

from openai import OpenAI
openai_client = OpenAI()

search_results = json.dumps(results, indent=2)
query = 'python function definition'




In [70]:
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_results}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        query=question,
        search_results=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [71]:
answer = rag('python function definition')
print(answer)

To define a function in Python, you follow a specific syntax. Here’s a simple outline you can use:

1. **Function Definition**: Start with the `def` keyword, followed by the name of the function and parentheses `()`.
2. **Parameters**: If your function takes parameters (inputs), list them inside the parentheses.
3. **Colon**: End the header with a colon `:`.
4. **Function Body**: Indent the body of the function, containing the statements that define what the function does.
5. **Return Statement**: Optionally, use the `return` statement to send a result back to the caller.

### Example
Here’s an example of a simple function that adds two numbers:

```python
def add(a, b):
    result = a + b
    return result
```

You can call this function with two arguments:

```python
sum = add(5, 3)  # sum will be 8
```

### Key Points
- Functions create their own local scope, so variable names inside the function do not affect the variables outside it, addressing the issue of name collisions.
- You 

In [72]:
def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    usage = response.usage
    return response.output_text, usage.input_tokens, usage.output_tokens

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, input_tokens, output_tokens = llm(prompt, instructions)
    return answer, input_tokens, output_tokens

In [73]:
answer, input_tokens, output_tokens = rag('python function definition')
print(answer)
print("input tokens:", input_tokens)
print("output tokens:", output_tokens)

In Python, a function is defined using the `def` keyword, followed by the function name and parentheses that may include parameters. The general structure is:

```python
def function_name(parameters):
    # code to execute
    return result
```

### Example:

Here's a simple example of defining a function that adds two numbers:

```python
def add_numbers(a, b):
    result = a + b
    return result
```

- **Function Name**: `add_numbers`
- **Parameters**: `a` and `b`
- **Return Value**: The sum of `a` and `b`

### Calling the Function:

You can call the function like this:

```python
sum_value = add_numbers(3, 5)
print(sum_value)  # Output: 8
```

### Notes:
- Functions have their own scope, so variables defined within the function do not affect variables outside it.
- You can pass multiple parameters to a function, and it can return multiple values as a tuple if needed.

This structure allows for modular code that can be reused, making your programming more efficient and organized.
inp

In [74]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")



In [75]:
def llm_structured(user_prompt, instructions, output_type, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    usage = response.usage
    return response.output_parsed, usage.input_tokens, usage.output_tokens

def rag_structured(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, input_tokens, output_tokens = llm_structured(prompt, instructions, RAGResponse)
    return answer, input_tokens, output_tokens

In [60]:
q = 'python function definition'

_, plain_in, plain_out = rag(q)
structured, struct_in, struct_out = rag_structured(q)

print("unstructured input tokens:", plain_in)
print("structured input tokens:", struct_in)
print("difference:", struct_in - plain_in)

print(structured)

unstructured input tokens: 7928
structured input tokens: 8114
difference: 186
answer="### Python Function Definition\n\nIn Python, a function is defined using the `def` keyword followed by the function name and parentheses containing any parameters. Here’s a basic structure of a function:\n\n```python\ndef function_name(parameters):\n    # Function body\n    # Perform actions\n    return result   # Optional\n```\n\n### Example:\nHere's an example of a function that adds two numbers:\n\n```python\ndef add_numbers(a, b):  \n    result = a + b  \n    return result  \n```\n\nYou would call this function by using its name followed by the arguments in parentheses:\n```python\nsum = add_numbers(3, 5)  # sum will be 8\n```\n\nThis definition allows for encapsulating reusable code, where the function can be called multiple times with different arguments without polluting the main workspace with variables defined inside the function." found_answer=True confidence=0.95 confidence_explanation='The